# NCA Workbench

Notebook для серверного запуска baseline NCA.

- На диске в v1 только один видимый канал `density`: `[T, H, W, 1]`
- В модели состояние расширяется до `[B, 2, H, W]`
- `channel 0` — наблюдаемый density
- `channel 1` — latent hidden state, который создаётся внутри модели
- Это intentionally minimalist baseline; следующий шаг — `1 + hidden_dim`


In [ ]:
from pathlib import Path

import numpy as np
import torch

from NCA.dataset import build_dataloaders, discover_npy_files
from NCA.model import NCA
from NCA.train import deterministic_eval, stochastic_eval, train_epoch
from NCA.utils import build_initial_state, get_device, load_checkpoint, rollout_model, save_checkpoint, set_seed
from NCA.visualize import plot_metric_curves, plot_triptych, plot_uncertainty_heatmap, save_rollout_animation


In [ ]:
# Все пути относительные к корню репозитория.
CONFIG = {
    'project_root': Path('.'),
    'data_root': Path('NCA/data'),
    'pattern': '**/*.npy',
    'run_dir': Path('NCA/runs/workbench'),
    'split_mode': 'within_file',
    'split_ratios': (0.8, 0.1, 0.1),
    'train_steps': (4, 16),
    'eval_steps': {'one_step': 1, 'rollout': 16, 'stochastic': 16},
    'batch_size': 8,
    'epochs': 5,
    'lr': 1e-3,
    'kernel_size': 3,
    'model_width': 64,
    'update_prob': 0.5,
    'num_rollouts': 8,
    'loss_mode': 'hybrid',
    'lambda_intermediate': 0.5,
    'lambda_hidden_l2': 1e-4,
    'use_alive_mask': False,
    'seed': 0,
}

CONFIG['run_dir'].mkdir(parents=True, exist_ok=True)
set_seed(CONFIG['seed'])
device = get_device()
device


In [ ]:
files = discover_npy_files(CONFIG['data_root'], CONFIG['pattern'])
print(f'Found {len(files)} files')
sample = np.load(files[0])
print('Example shape:', sample.shape)
print('Expected disk format [T, H, W, F_data], with F_data=1 in v1')


In [ ]:
loaders, normalizer = build_dataloaders(
    data_root=CONFIG['data_root'],
    pattern=CONFIG['pattern'],
    split_mode=CONFIG['split_mode'],
    split_ratios=CONFIG['split_ratios'],
    train_steps=CONFIG['train_steps'],
    eval_steps=CONFIG['eval_steps'],
    batch_size=CONFIG['batch_size'],
    eval_batch_size=CONFIG['batch_size'],
    seed=CONFIG['seed'],
)
train_batch = next(iter(loaders['train']))
train_batch['input_visible'].shape, train_batch['targets_visible'].shape


In [ ]:
model = NCA(
    state_channels=2,
    model_width=CONFIG['model_width'],
    kernel_size=CONFIG['kernel_size'],
    update_prob=CONFIG['update_prob'],
    use_alive_mask=CONFIG['use_alive_mask'],
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
model


In [ ]:
history = []
for epoch in range(CONFIG['epochs']):
    train_metrics = train_epoch(
        model,
        loaders['train'],
        optimizer,
        device=device,
        loss_mode=CONFIG['loss_mode'],
        lambda_intermediate=CONFIG['lambda_intermediate'],
        lambda_hidden_l2=CONFIG['lambda_hidden_l2'],
    )
    val_det = deterministic_eval(model, loaders['val_rollout'], device=device)
    val_stoch = stochastic_eval(model, loaders['val_rollout'], device=device, num_rollouts=CONFIG['num_rollouts'])
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_det_{k}': v for k, v in val_det.items()}, **{f'val_stoch_{k}': v for k, v in val_stoch.items()}}
    history.append(row)
    print(row)

save_checkpoint(CONFIG['run_dir'] / 'checkpoint_latest.pt', model, optimizer, CONFIG['epochs'] - 1, {'config': CONFIG, 'normalizer': normalizer.state_dict()}, history[-1])
history[-1]


In [ ]:
plot_metric_curves(history, CONFIG['run_dir'] / 'loss_curve.png', ['train_loss', 'val_det_rollout_mse', 'val_stoch_expected_mse'])
CONFIG['run_dir'] / 'loss_curve.png'


In [ ]:
batch = next(iter(loaders['val_rollout']))
visible = batch['input_visible'].to(device)
state0 = build_initial_state(visible, hidden_channels=1, hidden_init='zeros')
det_rollout = rollout_model(model, state0, steps=int(batch['horizons'].max().item()), stochastic=False)
stoch_rollouts = torch.stack([rollout_model(model, state0, steps=int(batch['horizons'].max().item()), stochastic=True) for _ in range(CONFIG['num_rollouts'])], dim=0)

plot_triptych(
    batch['input_visible'][0],
    batch['targets_visible'][0, -1],
    det_rollout[-1, 0, 0:1].detach().cpu(),
    CONFIG['run_dir'] / 'triptych.png',
    title='Visible density channel',
)
plot_uncertainty_heatmap(stoch_rollouts, CONFIG['run_dir'] / 'uncertainty.png')
save_rollout_animation(det_rollout[:, 0:1, 0:1].detach().cpu(), CONFIG['run_dir'] / 'det_rollout.gif')
CONFIG['run_dir'] / 'triptych.png'


## Baseline protocol

- `deterministic_eval(stochastic=False)` использовать как воспроизводимый benchmark между моделями.
- `stochastic_eval(stochastic=True, K rollouts)` использовать как вероятностную оценку динамики и разброса.
- Все публичные heatmap и основные метрики считаются только по видимому каналу `[..., 0]`.
